# Option-order consistency test (Qwen2.5-3B-Instruct, Phi-4-mini)

Each test item is scored under **all 6 orders of its options** (the first order is the
original prompt, so it must reproduce the baseline accuracy). A model that reads the maths
picks the same option wherever it appears; a model with letter bias follows the letter.

Columns of the final table, per slice:
- `acc orig` - accuracy in the original order (= the baseline number)
- `acc mean6` - accuracy averaged over the 6 orders (letter bias averages out)
- `robust` - correct under **all 6** orders (pure guessing: 0.14%)
- `content` - same option picked under all 6 orders (right or wrong)
- `letter` - same letter picked under all 6 orders (pure position bias)

Needs on Drive: `option_permutations.py`, `baseline_letter_logprobs.py`, `ivp_mcq/`.
Runtime: **T4 GPU**. About 5-10 min per model (792 items x 6 orders).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Thesis'   # <- edit if needed
%cd {PROJECT_DIR}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install "transformers>=4.51" accelerate

In [ ]:
SPLIT = 'test'
RUNS = {                                   # model key -> system prompt (None = prompt v1)
    'qwen2.5-3b-instruct': None,
    'phi-4-mini-instruct': 'letter-only',
}
BATCH_SIZE = 8

In [ ]:
import subprocess, shlex
failed = []
for m, sysm in RUNS.items():
    cmd = f'python option_permutations.py --model {m} --split {SPLIT} --batch-size {BATCH_SIZE}'
    if sysm:
        cmd += ' --system ' + shlex.quote(sysm)
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = [l for l in r.stdout.splitlines() if 'acc=' in l]
    print(out[-1] if out else '', flush=True)
    if r.returncode:
        failed.append(m)
        print('FAILED:' + chr(10) + chr(10).join(r.stderr.strip().splitlines()[-8:]), flush=True)
print('failed runs:', failed or 'none')

In [ ]:
!python option_permutations.py --table "results/option_permutations/*_{SPLIT}/summary.json"